# SIH26158 - Single-Pass Drone Video to Georeferenced 3D Model

This notebook runs the complete SIH pipeline on a Colab GPU. Inputs are one 1080p/4K drone video and its synchronized DJI SRT or normalized GPS CSV. Outputs include PLY, LAS, GLB, OBJ, GeoTIFF, georeferencing metadata, timings, and accuracy diagnostics.

Before starting, select **Runtime > Change runtime type > T4 GPU**. Upload this project folder to `MyDrive/SIH26158/`, with input files under `MyDrive/SIH26158/input/`.

In [ ]:
# CELL 1 - verify GPU and mount Google Drive
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
assert gpu.returncode == 0, 'No GPU detected. Select a T4 GPU runtime before continuing.'
print('GPU:', gpu.stdout.strip())
PROJECT_DIR = '/content/drive/MyDrive/SIH26158'
assert os.path.isfile(f'{PROJECT_DIR}/requirements.txt'), f'Project not found at {PROJECT_DIR}'
print('Project:', PROJECT_DIR)

In [ ]:
# CELL 2 - install Conda. This intentionally restarts the runtime once.
# After the restart, continue from CELL 3; do not rerun this cell.
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# CELL 3 - install the CUDA COLMAP build and Python dependencies
import os, shutil, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/SIH26158'
assert shutil.which('mamba'), 'Conda is missing. Run CELL 2 and wait for its runtime restart.'
pin = '/usr/local/conda-meta/pinned'
if os.path.exists(pin): os.remove(pin)
install = subprocess.run(['mamba','install','-y','-q','-c','conda-forge','colmap','libfaiss','openimageio'], text=True)
assert install.returncode == 0, 'COLMAP installation failed'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{PROJECT_DIR}/requirements.txt'], check=True)
check = subprocess.run(['colmap','-h'], capture_output=True, text=True, errors='replace')
header = (check.stdout or '') + (check.stderr or '')
print('\n'.join(header.splitlines()[:8]))
assert check.returncode == 0 and 'COLMAP' in header, 'COLMAP does not run'
assert 'CUDA' in header, 'This COLMAP build does not report CUDA support'
print('Installation complete')

In [ ]:
# CELL 4 - configure this flight
#@markdown ### Fill in the two paths below, then run this cell -- no code editing needed. Any of these work for each field:
#@markdown - A path already in your mounted Drive, e.g. `/content/drive/MyDrive/MyFolder/video.mp4`
#@markdown - A Google Drive **share link** (`drive.google.com/file/d/...`) -- set sharing to "Anyone with the link" first
#@markdown - A direct http(s) URL (Hugging Face `resolve/...` links, etc.)
#@markdown
#@markdown Leave RUN_NAME blank to auto-name the output folder from the video filename + a timestamp (always unique, never overwrites a previous run). Set it explicitly only when you deliberately want to resume/reuse that same output folder.
RUN_NAME = "" #@param {type:"string"}
VIDEO_SOURCE = "" #@param {type:"string"}
TELEMETRY_SOURCE = "" #@param {type:"string"}
#@markdown PROFILE = deadline-adaptive is the 15-minute-budget profile (v14: GPS-anchored incremental SfM with a 70-degree default lens, coverage-guided dense depth maps capped at 96 / 35% of frames, exact UTM export). PROFILE = auto reads this flight's telemetry and picks the keyframe selector itself (adaptive for flights with large altitude changes). TARGET_FRAMES = 0 uses the validated 120-frame budget; type a number to override it (more frames = longer run, by an amount that varies per scene).
TARGET_FRAMES = 0 #@param {type:"integer"}
DSM_RESOLUTION_M = 0.5 #@param {type:"number"}
PROFILE = "deadline-adaptive" #@param ["deadline-adaptive", "auto", "verified-fast", "verified-fast-adaptive"]
#@markdown Adaptive keyframe screening (PROFILE=verified-fast-adaptive) does exhaustive
#@markdown pairwise visual verification across every candidate frame and must finish
#@markdown within this wall-clock budget or the run fails with `TimeoutError: Visual
#@markdown matching exceeded screening deadline`. The library default is only 180s,
#@markdown which is too short for anything but very short clips -- raise this for
#@markdown longer videos (roughly: a few minutes per minute of footage).
ADAPTIVE_SCREENING_TIMEOUT_S = 1800 #@param {type:"number"}
#@markdown TELEMETRY_ALTITUDE_OFFSET_M: DJI `rel_alt` heights are measured from the takeoff point, not sea level.
#@markdown If the run prints `NOTE: ... relative to takeoff`, type the takeoff ground elevation in metres here
#@markdown (from Google Earth) so DSM heights become absolute. Leave 0 when the telemetry is already absolute.
TELEMETRY_ALTITUDE_OFFSET_M = 0.0 #@param {type:"number"}

import os, sys, csv, json, datetime, urllib.request
from pathlib import Path

RUN_NAME = RUN_NAME.strip()
VIDEO_SOURCE = VIDEO_SOURCE.strip()
TELEMETRY_SOURCE = TELEMETRY_SOURCE.strip()

PROJECT_DIR = '/content/drive/MyDrive/SIH26158'

if not RUN_NAME.strip():
    _base_name = Path(VIDEO_SOURCE.split('?')[0]).stem or 'flight'
    _timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    RUN_NAME = f'{_base_name}_{_timestamp}'
    print(f'RUN_NAME left blank -> auto-generated: {RUN_NAME}')

LOCAL_DIR = f'/content/input_{RUN_NAME}'
os.makedirs(LOCAL_DIR, exist_ok=True)

def _resolve(source, local_name):
    """Accept a Drive-mounted path, a Google Drive share link, or a direct
    http(s) URL. A share link needs Drive's own export mechanism (gdown) --
    a plain urllib fetch on drive.google.com returns the HTML preview page,
    not the file, which is the bug this replaces."""
    local_path = f'{LOCAL_DIR}/{local_name}'
    if 'drive.google.com' in source:
        if not os.path.isfile(local_path):
            try:
                import gdown
            except ImportError:
                import subprocess, sys
                subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
                import gdown
            print(f'Fetching Drive share link: {source}')
            gdown.download(url=source, output=local_path, quiet=False, fuzzy=True)
        return local_path
    if source.startswith('http://') or source.startswith('https://'):
        if not os.path.isfile(local_path):
            print(f'Downloading {source} ...')
            urllib.request.urlretrieve(source, local_path)
        return local_path
    return source  # already a local/Drive-mounted path -- use as-is

_video_ext = Path(VIDEO_SOURCE.split('?')[0]).suffix or '.mp4'
VIDEO = _resolve(VIDEO_SOURCE, f'video{_video_ext}')
_telemetry_ext = Path(TELEMETRY_SOURCE.split('?')[0]).suffix or '.csv'
telemetry_raw = _resolve(TELEMETRY_SOURCE, f'telemetry_raw{_telemetry_ext}')

# Load the pipeline package first: a freshly re-extracted zip is picked up
# without restarting the runtime once earlier imports are dropped.
sys.path.insert(0, PROJECT_DIR)
for _m in [m for m in sys.modules if m == 'sih_drone_pipeline' or m.startswith('sih_drone_pipeline.')]:
    del sys.modules[_m]
from sih_drone_pipeline.telemetry import is_dji_flight_record, normalize_dji_flight_record

def _video_duration_s(path):
    import subprocess
    probe = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                            '-of', 'default=nw=1:nk=1', path], capture_output=True, text=True)
    try:
        return float(probe.stdout.strip())
    except ValueError:
        import cv2
        capture = cv2.VideoCapture(path)
        frames, fps = capture.get(cv2.CAP_PROP_FRAME_COUNT), capture.get(cv2.CAP_PROP_FPS)
        capture.release()
        return frames / fps

# A raw DJI flight record is timed by flight time (seconds since takeoff), not
# video time, and may hold several recordings. Normalizing puts it on the video
# clock using the recording whose length matches this video -- without this every
# GPS fix lands on the wrong frame. A clean CSV or .srt passes through untouched.
TELEMETRY = telemetry_raw
TELEMETRY_SYNC = None
if telemetry_raw.lower().endswith('.csv') and is_dji_flight_record(telemetry_raw):
    TELEMETRY = f'{LOCAL_DIR}/telemetry_normalized.csv'
    TELEMETRY_SYNC = normalize_dji_flight_record(telemetry_raw, TELEMETRY, _video_duration_s(VIDEO))
    print(f"Detected raw DJI flight record -> {TELEMETRY_SYNC['rows']} GPS fixes on the video clock")
    print(f"  video starts at flight time {TELEMETRY_SYNC['video_start_flytime_s']:.1f} s "
          f"(recordings in this record: {TELEMETRY_SYNC['recording_segments_flytime_s']})")

# Classify this flight from its own telemetry -- no video needed -- and let
# that choose the processing profile / frame budget when set to auto.
from dataclasses import asdict
from sih_drone_pipeline.telemetry import load_telemetry
from sih_drone_pipeline.flight_classifier import classify
FLIGHT = classify(load_telemetry(TELEMETRY))
print('--- Flight classification (from telemetry) ---')
print(f'  footprint diagonal   : {FLIGHT.area_diagonal_m:.0f} m')
print(f'  altitude range       : {FLIGHT.altitude_range_m:.1f} m')
print(f'  viewpoint complexity : {FLIGHT.viewpoint_complexity_ratio*100:.2f} %  (adaptive above 5 %)')
for _line in FLIGHT.reasoning:
    print('  *', _line)
if PROFILE == 'auto':
    PROFILE = FLIGHT.recommended_profile
    print(f'PROFILE auto -> {PROFILE}')
if TARGET_FRAMES is None or TARGET_FRAMES <= 0:
    TARGET_FRAMES = FLIGHT.recommended_target_frames
    print(f'TARGET_FRAMES auto -> {TARGET_FRAMES}')
print('---')

QUALITY = 'draft'                               # draft first; full for final output
GPS_PRIOR_HORIZONTAL_STD_M = 2.0                # tighter than the 5 m default; anchors the mapper against telemetry harder
GPS_PRIOR_VERTICAL_STD_M = 3.0                  # counters single-strip SfM drift/bow on long constant-altitude flights
CAMERA_CALIBRATION = ''                         # optional calibrated camera JSON
GCP_CONTROL_POINTS = ''                         # optional GCP correction CSV
VALIDATION_CHECKPOINTS = ''                     # independent holdout checkpoint CSV
VALIDATION_DISTANCES = ''                       # optional CSV with surveyed distances
GROUND_TRUTH_TRAJECTORY = ''                    # optional time_s,x_m,y_m,z_m CSV
AI_MASK_DYNAMIC_OBJECTS = True                  # YOLO segmentation: people/vehicles/animals
STRICT_CAPTURE_QUALITY = False                  # True blocks risky captures before COLMAP
RESET_WORKSPACE = True                          # clean previous temporary reconstruction
WORKSPACE = f'/content/sih26158_{RUN_NAME}'     # fast ephemeral disk
OUTPUT = f'{PROJECT_DIR}/outputs/{RUN_NAME}'    # persistent Drive output
os.makedirs(OUTPUT, exist_ok=True)
with open(f'{OUTPUT}/flight_classification.json', 'w', encoding='utf-8') as _f:
    json.dump({**asdict(FLIGHT), 'applied_profile': PROFILE, 'applied_target_frames': TARGET_FRAMES,
               'telemetry_sync': TELEMETRY_SYNC}, _f, indent=2)
assert os.path.isfile(VIDEO), f'Video not found: {VIDEO}'
assert os.path.isfile(TELEMETRY), f'Telemetry not found: {TELEMETRY}'
print('Video:', VIDEO)
print('Telemetry:', TELEMETRY)
print('Output:', OUTPUT)

In [ ]:
# CELL 5 - preflight, GPU monitoring, complete pipeline, and artifact verification
import os, subprocess, sys, time, shutil, json, glob
command = [sys.executable, '-m', 'sih_drone_pipeline', 'run',
           '--video', VIDEO, '--telemetry', TELEMETRY,
           '--workspace', WORKSPACE, '--output', OUTPUT,
           '--quality', QUALITY, '--profile', PROFILE, '--target-frames', str(TARGET_FRAMES),
           '--dsm-resolution', str(DSM_RESOLUTION_M),
           '--gps-prior-horizontal-std-m', str(GPS_PRIOR_HORIZONTAL_STD_M),
           '--gps-prior-vertical-std-m', str(GPS_PRIOR_VERTICAL_STD_M),
           '--adaptive-screening-timeout-s', str(ADAPTIVE_SCREENING_TIMEOUT_S)]
if TELEMETRY_ALTITUDE_OFFSET_M:
    command += ['--telemetry-altitude-offset-m', str(TELEMETRY_ALTITUDE_OFFSET_M)]
if AI_MASK_DYNAMIC_OBJECTS: command.append('--ai-mask-dynamic')
if STRICT_CAPTURE_QUALITY: command.append('--strict-capture-quality')
if CAMERA_CALIBRATION:
    command += ['--camera-calibration', CAMERA_CALIBRATION]
if GCP_CONTROL_POINTS:
    command += ['--gcp-control-points', GCP_CONTROL_POINTS]
if VALIDATION_CHECKPOINTS:
    command += ['--validation-checkpoints', VALIDATION_CHECKPOINTS]
if VALIDATION_DISTANCES:
    command += ['--validation-distances', VALIDATION_DISTANCES]
if GROUND_TRUTH_TRAJECTORY:
    command += ['--ground-truth-trajectory', GROUND_TRUTH_TRAJECTORY]
env = os.environ.copy()
env['PYTHONPATH'] = PROJECT_DIR + os.pathsep + env.get('PYTHONPATH', '')
os.makedirs(OUTPUT, exist_ok=True)
if RESET_WORKSPACE and os.path.isdir(WORKSPACE): shutil.rmtree(WORKSPACE)
preflight = [sys.executable, '-m', 'sih_drone_pipeline', 'preflight', '--video', VIDEO, '--telemetry', TELEMETRY, '--workspace', WORKSPACE, '--output', f'{OUTPUT}/preflight.json']
assert subprocess.run(preflight, cwd=PROJECT_DIR, env=env).returncode == 0, 'Preflight failed; open preflight.json'
stop_file = '/content/sih_gpu_monitor.stop'
if os.path.exists(stop_file): os.remove(stop_file)
monitor = subprocess.Popen([sys.executable, '-m', 'sih_drone_pipeline.gpu_monitor', '--output', f'{OUTPUT}/gpu_usage.csv', '--stop-file', stop_file], cwd=PROJECT_DIR, env=env)
print(' '.join(command))
started = time.time()
# Streamed AND saved: subprocess.run() alone would only stream to the cell,
# so a failure's real error scrolls away unrecovered once the cell reruns.
# Popen + line-by-line tee keeps the exact same live output but also writes
# it to OUTPUT/pipeline_stdout.log, so a failure is diagnosable after the fact.
pipeline_log_path = f'{OUTPUT}/pipeline_stdout.log'
try:
    with open(pipeline_log_path, 'w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(
            command, cwd=PROJECT_DIR, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in proc.stdout:
            sys.stdout.write(line)
            log_file.write(line)
        proc.wait()
    result_returncode = proc.returncode
finally:
    open(stop_file, 'w').close()
    monitor.wait(timeout=15)
    if os.path.isdir(f'{WORKSPACE}/logs'): shutil.copytree(f'{WORKSPACE}/logs', f'{OUTPUT}/logs', dirs_exist_ok=True)
    if os.path.isfile(f'{WORKSPACE}/run_report.partial.json'): shutil.copy2(f'{WORKSPACE}/run_report.partial.json', f'{OUTPUT}/run_report.partial.json')
print(f'Total notebook elapsed time: {(time.time()-started)/60:.1f} minutes')

if result_returncode != 0:
    print('\n=== PIPELINE FAILED -- diagnostic summary ===')
    print(f'Full captured log: {pipeline_log_path}')
    with open(pipeline_log_path, encoding='utf-8', errors='replace') as f:
        tail = f.readlines()[-40:]
    print('--- last 40 lines of pipeline output ---')
    print(''.join(tail))
    partial = f'{OUTPUT}/run_report.partial.json'
    if os.path.isfile(partial):
        print('\n--- run_report.partial.json (last stage reached before failure) ---')
        print(json.dumps(json.load(open(partial)), indent=2)[:3000])
    else:
        print('\nNo run_report.partial.json was written -- the pipeline failed before COLMAP '
              'reconstruction started (during keyframe extraction, telemetry sync, or AI masking).')
    logs_dir = f'{OUTPUT}/logs'
    if os.path.isdir(logs_dir):
        print(f'\nAlso inspect stage logs under: {logs_dir}')

assert result_returncode == 0, f'Pipeline failed; see {pipeline_log_path}, {OUTPUT}/logs, and run_report.partial.json above'
verify = [sys.executable, '-m', 'sih_drone_pipeline', 'verify', '--output', OUTPUT]
assert subprocess.run(verify, cwd=PROJECT_DIR, env=env).returncode == 0, 'Artifact verification failed; open verification_report.json'

In [ ]:
# CELL 6 - inspect the judging metrics and output files
import json, os, glob
report = json.load(open(f'{OUTPUT}/run_report.json'))
verification = json.load(open(f'{OUTPUT}/verification_report.json'))
gpu = json.load(open(f'{OUTPUT}/gpu_usage.summary.json'))
print(json.dumps({
    'registered_images': report.get('sparse_metrics', {}).get('registered_images'),
    'mean_reprojection_error_px': report.get('sparse_metrics', {}).get('mean_reprojection_error_px'),
    'processing_minutes': round(report.get('wall_clock_seconds', 0) / 60, 2),
    'gps_validation': report.get('validation'),
    'targets': report.get('targets'),
    'artifact_checks_pass': verification.get('artifact_checks_pass'),
    'quality_checks': verification.get('quality_checks'),
    'production_ready': verification.get('production_ready'),
    'gpu_overall': gpu,
    'gpu_by_stage': verification.get('gpu_by_stage'),
}, indent=2))
print('\nOutputs:')
for path in sorted(glob.glob(f'{OUTPUT}/*')):
    print(f'  {os.path.basename(path):28s} {os.path.getsize(path)/1024/1024:9.1f} MB')

In [ ]:
# CELL 7 - v14 run summary (paste this output back for the 3-dataset comparison)
import json, os

def _load(name):
    path = f'{OUTPUT}/{name}'
    return json.load(open(path, encoding='utf-8')) if os.path.isfile(path) else {}

def _find(obj, key):
    """First value stored under `key` anywhere in a nested report."""
    if isinstance(obj, dict):
        if key in obj:
            return obj[key]
        for value in obj.values():
            found = _find(value, key)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = _find(value, key)
            if found is not None:
                return found
    return None

report = _load('run_report.json')
verification = _load('verification_report.json')
georef = _load('georeference.json')

print('RUN_NAME:', RUN_NAME)
print(f"wall clock: {report.get('wall_clock_seconds', 0) / 60:.1f} min")
stages = _find(report, 'stages') or {}
for name, stage in stages.items():
    if isinstance(stage, dict) and 'seconds' in stage:
        print(f'  {name:<40} {stage["seconds"] / 60:6.2f} min')

sparse = _find(report, 'sparse_metrics') or {}
print('sparse:', {k: sparse.get(k) for k in ('registered_images', 'registration_percent',
                                             'mean_reprojection_error_px') if k in sparse})
selection = _find(report, 'dense_reference_selection') or {}
print('depth-map selection:', {k: selection.get(k) for k in (
    'selection_mode_used', 'selected_references', 'total_references') if k in selection})
if selection.get('coverage'):
    print('  coverage detail:', selection['coverage'])
print('footprint DSM coverage:', _find(verification, 'footprint_coverage'))
print('old bbox valid_fraction:', _find(verification, 'valid_fraction'))
print('horizontal conversion:', georef.get('horizontal_conversion'))
print('vertical reference:', georef.get('vertical_reference'))
print('GPS residual RMSE (m):', report.get('validation', {}).get('gps_alignment_rmse_m'))
gates = verification.get('quality_checks', {})
print('failed quality gates:', [k for k, v in gates.items() if v is not True] or 'none')
print('terminal state:', verification.get('terminal_state'), verification.get('terminal_reasons'))


## Quality gates before presenting

- Most extracted frames should be registered.
- Mean reprojection error should normally be below 1 pixel.
- GPS alignment residual should be inspected, but it is not independent proof of accuracy.
- Add surveyed distances using `examples/validation_distances.example.csv`; only those checks can prove surface accuracy.
- The draft profile must be timed on the final 10-minute test video.
- Download `model.glb` and `viewer_metadata.json`, then open both in the metric web viewer.